# Notebook 02: Data Preparation and Integration

**Project:** Pharmacogenomics Machine Learning

**Author:** Sofia Muñoz

**Purpose:** This notebook prepares the raw ClinPGx datasets for machine learning by analyzing dataset relationships, determining merge strategies, cleaning the data, engineering features, and producing a final master dataset suitable for model development.

**Datasets:**
- Clinical Variants
- Variant Drug Annotations
- Variant Phenotype Annotations
- Variant Functional Assay Annotations

## Objectives

By the end of this notebook, I will:
- Identify relationships among all four datasets.
- Determine scientifically appropriate merge keys.
- Validate candidate merge keys.
- Develop a documented merge strategy.
- Clean and standardize each dataset.
- Integrate the datasets into a single master table.
- Engineer features for machine learning.
- Export a processed dataset for model development.

## Table of Contents

1. Import Libraries
2. Load Raw Datasets
3. Standardize Column Names
4. Dataset Schema Harmonization
5. Dataset Relationship Analysis
6. Cross-Dataset Validation of Variant Annotation ID
7. Merge Strategy
8. Data Cleaning
9. Standardization
10. Missing Values
11. Duplicate Analysis
12. Dataset Integration
13. Feature Engineering
14. Final Quality Checks
15. Export Processed Dataset
16. Reflection

### 1. Import Libraries

In [953]:
# Import the pandas library for reading, manipulating, and analyzing tabular data.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import Matplotlib for creating graphs and visualizations.
import matplotlib.pyplot as plt

# Import Path from pathlib to build operating system-independent file paths.
from pathlib import Path

### 2. Load Raw Datasets

In [954]:
# Load the Clinical Variants dataset.
clinical_raw = pd.read_csv(
    "../data/raw/clinicalVariants.tsv",
    sep="\t"
)
# Load the Variant Drug Annotations dataset.
drug_raw = pd.read_csv(
    "../data/raw/var_drug_ann.tsv",
    sep="\t"
)
# Load the Variant Phenotype Annotations dataset.
phenotype_raw = pd.read_csv(
    "../data/raw/var_pheno_ann.tsv",
    sep="\t"
)
# Load the Variant Functional Assays Annotations dataset.
functional_raw = pd.read_csv(
    "../data/raw/var_fa_ann.tsv",
    sep="\t"
)

In [955]:
# Verify the files loaded
print("Clinical Variants:", clinical_raw.shape)
print("Drug Annotations:", drug_raw.shape)
print("Phenotype Annotations:", phenotype_raw.shape)
print("Functional Assays:", functional_raw.shape)

Clinical Variants: (5190, 6)
Drug Annotations: (12975, 22)
Phenotype Annotations: (14490, 25)
Functional Assays: (2153, 23)


In [956]:
# Create working copies of each dataset.
# All preprocessing will be performed on these copies, preserving the original raw datasets for reference.

clinical = clinical_raw.copy()
drug = drug_raw.copy()
phenotype = phenotype_raw.copy()
functional = functional_raw.copy()

In [957]:
# Verify the copies
print(clinical.shape == clinical_raw.shape)
print(drug.shape == drug_raw.shape)
print(phenotype.shape == phenotype_raw.shape)
print(functional.shape == functional_raw.shape)

True
True
True
True


### 3. Standardize Column Names

The column names of the working datasets are standardized to a consistent
snake_case convention. This improves readability and prevents inconsistencies
when referring to columns throughout the preprocessing and integration
pipeline.

Only the working copies are modified. The original raw DataFrames remain
unchanged.

In [958]:
# Standardize column names across all working datasets.
# The transformation:
#   1. Removes leading and trailing whitespace.
#   2. Converts all characters to lowercase.
#   3. Replaces spaces with underscores.

def standardize_column_names(df):
    """Return a DataFrame with standardized snake_case column names."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

# Apply the standardization function to each working dataset.
clinical = standardize_column_names(clinical)
drug = standardize_column_names(drug)
phenotype = standardize_column_names(phenotype)
functional = standardize_column_names(functional)

In [959]:
# Display the standardized column names for each dataset.

print("Clinical Variants:")
print(clinical.columns.tolist())

print("\nVariant Drug Annotations:")
print(drug.columns.tolist())

print("\nVariant Phenotype Annotations:")
print(phenotype.columns.tolist())

print("\nVariant Functional Assay Annotations:")
print(functional.columns.tolist())

Clinical Variants:
['variant', 'gene', 'type', 'level_of_evidence', 'chemicals', 'phenotypes']

Variant Drug Annotations:
['variant_annotation_id', 'variant_haplotypes', 'gene', 'drug_s', 'pmid', 'phenotype_category', 'significance', 'notes', 'sentence', 'alleles', 'specialty_population', 'metabolizer_types', 'isplural', 'is_is_not_associated', 'direction_of_effect', 'pd_pk_terms', 'multiple_drugs_and_or', 'population_types', 'population_phenotypes_or_diseases', 'multiple_phenotypes_or_diseases_and_or', 'comparison_allele_s_or_genotype_s', 'comparison_metabolizer_types']

Variant Phenotype Annotations:
['variant_annotation_id', 'variant_haplotypes', 'gene', 'drug_s', 'pmid', 'phenotype_category', 'significance', 'notes', 'sentence', 'alleles', 'specialty_population', 'metabolizer_types', 'isplural', 'is_is_not_associated', 'direction_of_effect', 'side_effect_efficacy_other', 'phenotype', 'multiple_phenotypes_and_or', 'when_treated_with_exposed_to_when_assayed_with', 'multiple_drugs_and

### 4. Dataset Schema Harmonization

Standardizing column names improves consistency within the computational
pipeline, but similarly named columns are not assumed to represent the
same biological concept.

Potentially equivalent columns are therefore evaluated based on their
definitions, contents, data types, missingness, and biological meaning
before being harmonized for dataset integration.

#### 4.1 'variant' vs. 'variant_haplotype'

In [960]:
# Compare example values from the potentially equivalent variant columns.

print("Clinical Variants:")
print(clinical["variant"].dropna().head(10))

print("\nVariant Drug Annotations:")
print(drug["variant_haplotypes"].dropna().head(10))

print("\nVariant Phenotype Annotations:")
print(phenotype["variant_haplotypes"].dropna().head(10))

print("\nVariant Functional Assay Annotations:")
print(functional["variant_haplotypes"].dropna().head(10))

Clinical Variants:
0                        CYP2C9*1, CYP2C9*3, CYP2C9*13
1                                           rs17376848
2                                            rs2297595
3                                            rs1801265
4                      CYP2C19*1, CYP2C19*2, CYP2C19*3
5    CYP2C9*1, CYP2C9*2, CYP2C9*3, CYP2C9*5, CYP2C9...
6                                            rs1801160
7                                            rs1801159
8    UGT1A1*1, UGT1A1*6, UGT1A1*28, UGT1A1*36, UGT1...
9            TPMT*1, TPMT*2, TPMT*3A, TPMT*3B, TPMT*3C
Name: variant, dtype: str

Variant Drug Annotations:
0     CYP3A4*1, CYP3A4*17
1               rs2909451
2                rs706795
3              rs16918842
4      CYP2C9*1, CYP2C9*3
5               rs2285676
6               CYP2C9*11
7                rs163184
8     CYP2B6*1, CYP2B6*18
9    CYP2C19*1, CYP2C19*2
Name: variant_haplotypes, dtype: str

Variant Phenotype Annotations:
0                    HLA-B*35:08
1               

In [961]:
print(clinical["variant"].dtype)
print(drug["variant_haplotypes"].dtype)
print(phenotype["variant_haplotypes"].dtype)
print(functional["variant_haplotypes"].dtype)

str
str
str
str


In [962]:
print(
    "Clinical:",
    clinical["variant"].isna().mean()
)
print(
    "Drug:",
    drug["variant_haplotypes"].isna().mean()
)
print(
    "Phenotype:",
    phenotype["variant_haplotypes"].isna().mean()
)
print(
    "Functional:",
    functional["variant_haplotypes"].isna().mean()
)

Clinical: 0.0
Drug: 0.0
Phenotype: 0.0
Functional: 0.0


In [963]:
# Create sets of non-missing variant representations.
clinical_variants = set(
    clinical["variant"].dropna()
)
drug_variants = set(
    drug["variant_haplotypes"].dropna()
)
phenotype_variants = set(
    phenotype["variant_haplotypes"].dropna()
)
functional_variants = set(
    functional["variant_haplotypes"].dropna()
)

In [964]:
# Calculate the number of exact variant representations shared between each pair of datasets.

print(
    "Clinical ∩ Drug:",
    len(clinical_variants & drug_variants)
)
print(
    "Clinical ∩ Phenotype:",
    len(clinical_variants & phenotype_variants)
)
print(
    "Clinical ∩ Functional:",
    len(clinical_variants & functional_variants)
)
print(
    "Drug ∩ Phenotype:",
    len(drug_variants & phenotype_variants)
)
print(
    "Drug ∩ Functional:",
    len(drug_variants & functional_variants)
)
print(
    "Phenotype ∩ Functional:",
    len(phenotype_variants & functional_variants)
)

Clinical ∩ Drug: 1825
Clinical ∩ Phenotype: 1964
Clinical ∩ Functional: 430
Drug ∩ Phenotype: 1367
Drug ∩ Functional: 337
Phenotype ∩ Functional: 361


*Harmonization of variant / variant_haplotypes Fields*

The Clinical Variants dataset contains a column named `variant`, while the
three annotation datasets contain a column named `variant_haplotypes`.
Inspection of the values in these fields demonstrated substantial
conceptual overlap, including rsIDs, star-allele representations, and
combinations of alleles or haplotypes.

Because the Clinical Variants `variant` field contains information that
fits within the broader variant/haplotype representation used by the
annotation datasets, the working Clinical Variants DataFrame will use
`variant_haplotypes` as the harmonized column name.

This rename is a schema harmonization step and does not imply that every
value has identical biological semantics or representation across all
datasets. The original column name and raw data are preserved in the
unaltered source dataset.

In [965]:
# Harmonize the Clinical Variants column name with the corresponding variant representation used in the annotation datasets.

clinical = clinical.rename(
    columns={"variant": "variant_haplotypes"})

In [966]:
print("variant" in clinical.columns)
print("variant_haplotypes" in clinical.columns)

False
True


#### 4.2 'chemicals' vs. 'drug_s'

In [967]:
print("Clinical Variants:")
print(clinical["chemicals"].dropna().head(10))

print("\nVariant Drug Annotations:")
print(drug["drug_s"].dropna().head(10))

Clinical Variants:
0                           lornoxicam
1                         capecitabine
2                         capecitabine
3                         capecitabine
4                      dexlansoprazole
5                             warfarin
6                         capecitabine
7                         capecitabine
8    atazanavir,atazanavir / ritonavir
9                         azathioprine
Name: chemicals, dtype: str

Variant Drug Annotations:
0                                           nifedipine
1                                          sitagliptin
2    citalopram, escitalopram, fluoxetine, fluvoxam...
3                                               heroin
4                                             warfarin
5                                          sitagliptin
6                                             warfarin
7                                          sitagliptin
8                                            efavirenz
9                 clomipramine, desmethyl

In [968]:
print(
    "Clinical chemical values:",
    clinical["chemicals"].nunique()
)
print(
    "Drug annotation drug values:",
    drug["drug_s"].nunique())

Clinical chemical values: 876
Drug annotation drug values: 1132


In [969]:
# Display values containing common delimiters that may indicate multiple chemicals in a single cell.

clinical.loc[
    clinical["chemicals"].astype(str).str.contains(
        r"[,;|]",
        regex=True,
        na=False),
    "chemicals"
].head(20)

8                      atazanavir,atazanavir / ritonavir
58                                    FOLFIRI,irinotecan
128                         simvastatin,simvastatin acid
133                           lovastatin,lovastatin acid
134                           lovastatin,lovastatin acid
144                         simvastatin,simvastatin acid
184    desflurane,enflurane,halothane,isoflurane,meth...
185    desflurane,enflurane,halothane,isoflurane,meth...
186    desflurane,enflurane,halothane,isoflurane,meth...
187    desflurane,enflurane,halothane,isoflurane,meth...
188    desflurane,enflurane,halothane,isoflurane,meth...
189    desflurane,enflurane,halothane,isoflurane,meth...
190    desflurane,enflurane,halothane,isoflurane,meth...
191    desflurane,enflurane,halothane,isoflurane,meth...
192    desflurane,enflurane,halothane,isoflurane,meth...
193    desflurane,enflurane,halothane,isoflurane,meth...
194    desflurane,enflurane,halothane,isoflurane,meth...
195    desflurane,enflurane,hal

*Harmonization of chemical / drug_s Fields*

The Clinical Variants dataset contains a `chemicals` column, while the
annotation datasets contain `drug_s` columns. Examination of the data
indicates that both fields can contain multiple values within a single
record and therefore have compatible structural representations.

The primary semantic difference is that `chemicals` has a broader scope
and can include substances such as nicotine and ethanol in addition to
medications. Because `chemicals` encompasses the full range of substances
represented by both fields, `chemicals` is selected as the provisional
canonical name for the integrated dataset.

The annotation datasets' `drug_s` columns will therefore be renamed to
`chemicals` in their working DataFrames. This harmonization preserves the
broader meaning of the source data rather than narrowing the canonical
schema to medications only.

As with other schema harmonization decisions, this rename does not alter
the underlying values or raw source files.

In [970]:
# Harmonize the chemical/drug column names across the working datasets.
# The broader term "chemicals" is used as the provisional canonical name.

drug = drug.rename(
    columns={"drug_s": "chemicals"}
)
phenotype = phenotype.rename(
    columns={"drug_s": "chemicals"}
)
functional = functional.rename(
    columns={"drug_s": "chemicals"}
)

#### 4.3 Canonical Schema Decisions

| Canonical field | Source fields | Decision | Rationale |
| :---: | :---: | :---: | :--- |
| `variant_haplotypes` | `variant`, `variant_haplotypes` | Harmonize | Both contain variant- and haplotype-related representations, including rsIDs and star-allele combinations. |
| `chemicals` | `chemicals`, `drugs` | Harmonize | Both represent chemical/substance exposures and can contain multiple values per record; `chemicals` has broader scope. |

### 5. Dataset Relationship Analysis

#### 5.1 Biological Roles of Each Dataset

| Dataset | What one row represents | Primary purpose |
| :--- | :--- | :--- |
| **Clinical Variants** | One curated clinical pharmacogenomic association describing how a specific genetic variant (or genotype/haplotype) influences a drug-related outcome (such as efficacy, toxicity, dosage, or metabolism) based on published clinical evidence. | Summarizes clinically actionable pharmacogenomic knowledge and recommendations for healthcare decision-making. |
| **Variant Drug Annotations** | One evidence record describing the relationship between a specific genetic variant and a particular drug, including the reported pharmacogenomic effect from an individual study or publication. Multiple rows may exist for the same variant because different drugs, studies, or evidence sources can be associated with it. | Provides detailed evidence linking genetic variants to drug response and serves as the primary source of variant–drug relationships. |
| **Variant Phenotype Annotations** | One evidence record describing how a specific genetic variant is associated with an observed phenotype (for example, altered metabolism, treatment response, or adverse drug reaction) reported in a publication. | Connects genetic variants with observed pharmacogenomic phenotypes that may explain differences in medication response. |
| **Variant Functional Assay Annotations** | One laboratory experimental result measuring the functional impact of a specific genetic variant on gene or protein activity. These data come from experimental assays rather than clinical observations. | Provides biological evidence about how variants affect molecular function, supporting interpretation of clinical and pharmacogenomic findings. |

#### 5.2 Candidate Shared Columns

| Column | Clinical | Drug | Phenotype | Functional |
| :--- | :---: | :---: | :---: | :---: |
| Variant Annotation ID | X | ✓ | ✓ | ✓ |
| Gene | ✓ | ✓ | ✓ | ✓ |
| Variant | ✓ | ✓ | ✓ | ✓ |
| Alleles | X | ✓ | ✓ | ✓ |
| Drug | ✓ | ✓ | ✓ | ✓ |
| PMID | X | ✓ | ✓ | ✓ |

##### Candidate Key 1: Variant Annotation ID

Variant Annotation ID is evaluated first because it appears in the three variant annotation datasets and is intended to uniquely identify a variant/drug annotation.

In [971]:
# Determine whether Variant Annotation ID exists in each dataset.

candidate_key1 = "variant_annotation_id"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key1 in df.columns}")

Clinical Variants: False
Variant Drug Annotations: True
Variant Phenotype Annotations: True
Variant Functional Assay Annotations: True


*Observation*

Variant Annotation ID is present in the Variant Drug Annotations, Variant Phenotype Annotations, and Variant Functional Assay Annotations datasets but is absent from the Clinical Variants dataset.

This suggests that it may serve as the primary integration key among the variant annotation datasets but cannot directly link the Clinical Variants dataset.

In [972]:
for name, df in datasets.items():
    if candidate_key1 in df.columns:
        missing = df[candidate_key1].isna().sum()
        print(f"{name}: {missing} missing values")

Variant Drug Annotations: 0 missing values
Variant Phenotype Annotations: 0 missing values
Variant Functional Assay Annotations: 0 missing values


In [973]:
for name, df in datasets.items():
    if candidate_key1 in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key1].is_unique)
        print()

Variant Drug Annotations
Unique: True

Variant Phenotype Annotations
Unique: True

Variant Functional Assay Annotations
Unique: True



In [974]:
summary1 = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
    ],
    "Answer": [
        "All but Clinical Variants",
        "Yes",
        "No",
    ]})
summary1

,Question,Answer
0,Present in datasets?,All but Clinical Variants
1,Unique?,Yes
2,Missing values?,No


##### Candidate Key 2: Gene

In [975]:
# Determine whether Variant Annotation ID exists in each dataset.

candidate_key2 = "gene"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key2 in df.columns}")

Clinical Variants: True
Variant Drug Annotations: True
Variant Phenotype Annotations: True
Variant Functional Assay Annotations: True


In [976]:
for name, df in datasets.items():
    if candidate_key2 in df.columns:
        missing = df[candidate_key2].isna().sum()
        print(f"{name}: {missing} missing values")

Clinical Variants: 254 missing values
Variant Drug Annotations: 340 missing values
Variant Phenotype Annotations: 416 missing values
Variant Functional Assay Annotations: 50 missing values


In [977]:
for name, df in datasets.items():
    if candidate_key2 in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key2].is_unique)
        print()

Clinical Variants
Unique: False

Variant Drug Annotations
Unique: False

Variant Phenotype Annotations
Unique: False

Variant Functional Assay Annotations
Unique: False



In [978]:
summary2 = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
    ],
    "Answer": [
        "All",
        "No",
        "Yes",
    ]})
summary2

,Question,Answer
0,Present in datasets?,All
1,Unique?,No
2,Missing values?,Yes


##### Candidate Key 3: Variant

In [979]:
# Determine whether Variant exists in each dataset.

candidate_key3 = "variant"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key3 in df.columns}")

Clinical Variants: False
Variant Drug Annotations: False
Variant Phenotype Annotations: False
Variant Functional Assay Annotations: False


In [980]:
for name, df in datasets.items():
    if candidate_key3 in df.columns:
        missing = df[candidate_key3].isna().sum()
        print(f"{name}: {missing} missing values")

In [981]:
for name, df in datasets.items():
    if candidate_key3 in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key3].is_unique)
        print()

In [982]:
summary3 = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
        "Represents biology?",
        "Merge candidate?"
    ],
    "Answer": [
        "All but Clinical Variants",
        "Yes",
        "No",
        "Yes",
        "Yes"
    ]})
summary3

,Question,Answer
0,Present in datasets?,All but Clinical Variants
1,Unique?,Yes
2,Missing values?,No
3,Represents biology?,Yes
4,Merge candidate?,Yes


### 6. Cross-Dataset Validation of Variant Annotation ID

The Variant Annotation ID is present in the Variant Drug Annotations,
Variant Phenotype Annotations, and Variant Functional Assays datasets.
Because the identifier appears to be a potentially useful relational
key, its cross-dataset behavior must be validated before it is used
for dataset integration.

This analysis investigates:

1. Whether Variant Annotation IDs are shared across datasets.
2. Whether shared IDs consistently represent the same gene.
3. Whether shared IDs consistently represent the same variant or haplotype.
4. Whether the same gene–variant–drug relationships occur across datasets.
5. Whether the datasets exhibit one-to-one, one-to-many, or many-to-many
   relationships that would affect the merge strategy.

The Clinical Variants dataset is excluded from the Variant Annotation ID
analysis because it does not contain this column.

#### 6.1 Presence of Variant Annotation ID

The first step is to confirm which datasets contain the Variant Annotation ID and therefore could potentially be connected through this identifier.

In [983]:
# Define the candidate merge key.
candidate_ID = "variant_annotation_id"

# Store the datasets in a dictionary so they can be examined consistently.
annotation_datasets = {
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assays Annotations": functional,
    "Clinical Variants": clinical
}

# Check whether the candidate key exists in each dataset.
for name, df in annotation_datasets.items():
    print(f"{name}: {candidate_ID in df.columns}")

Variant Drug Annotations: True
Variant Phenotype Annotations: True
Variant Functional Assays Annotations: True
Clinical Variants: False


*Observation*

Variant Annotation ID is present in the three variant annotation datasets but is
not present in the Clinical Variants dataset.

Therefore, any direct validation of this identifier is limited to the
Drug, Phenotype, and Functional Assay datasets.

#### 6.2 Completeness of Variant Annotation ID

A potential relational identifier should contain few or no missing values. Missing identifiers cannot be used to establish relationships between records.

In [984]:
# Count missing and non-missing Variant Annotation IDs in each annotation dataset.
for name, df in annotation_datasets.items():
    if candidate_ID in df.columns:
        missing = df[candidate_ID].isna().sum()
        present = df[candidate_ID].notna().sum()
        print(f"{name}")
        print(f"  Present: {present}")
        print(f"  Missing: {missing}")
        print()

Variant Drug Annotations
  Present: 12975
  Missing: 0

Variant Phenotype Annotations
  Present: 14490
  Missing: 0

Variant Functional Assays Annotations
  Present: 2153
  Missing: 0



#### 6.3 Uniqueness Within Each Dataset

A key that uniquely identifies a record should not occur multiple times within the same dataset. Duplicate identifiers may indicate either a one-to-many relationship or that the identifier represents a broader entity than an individual row.

In [985]:
# Examine the number of rows and unique Variant Annotation IDs in each annotation dataset.
for name, df in annotation_datasets.items():
    if candidate_ID in df.columns:
        total_rows = len(df)
        unique_ids = df[candidate_ID].nunique()
        print(name)
        print(f"  Total rows: {total_rows}")
        print(f"  Unique IDs: {unique_ids}")
        print(f"  Repeated IDs: {total_rows - unique_ids}")
        print()

Variant Drug Annotations
  Total rows: 12975
  Unique IDs: 12975
  Repeated IDs: 0

Variant Phenotype Annotations
  Total rows: 14490
  Unique IDs: 14490
  Repeated IDs: 0

Variant Functional Assays Annotations
  Total rows: 2153
  Unique IDs: 2153
  Repeated IDs: 0



#### 6.4 Cross-Dataset ID Overlap

If Variant Annotation ID represents the same annotation across multiple datasets, some identifiers should be expected to occur in more than one dataset.

The number of shared identifiers is therefore calculated for each pair of annotation datasets.

In [986]:
# Create a set containing the Variant Annotation IDs from each dataset.
drug_ids = set(drug[candidate_ID].dropna())
pheno_ids = set(phenotype[candidate_ID].dropna())
functional_ids = set(functional[candidate_ID].dropna())

# Calculate pairwise ID intersections.
drug_pheno_overlap = drug_ids & pheno_ids
drug_functional_overlap = drug_ids & functional_ids
pheno_functional_overlap = pheno_ids & functional_ids

print("Drug ↔ Phenotype:", len(drug_pheno_overlap))
print("Drug ↔ Functional:", len(drug_functional_overlap))
print("Phenotype ↔ Functional:", len(pheno_functional_overlap))

Drug ↔ Phenotype: 0
Drug ↔ Functional: 0
Phenotype ↔ Functional: 0


In [987]:
# Find Variant Annotation IDs shared by all three annotation datasets.
all_three_ids = drug_ids & pheno_ids & functional_ids
print("IDs shared across all three datasets:", len(all_three_ids))

IDs shared across all three datasets: 0


#### 6.5 Cross-Dataset Gene-Variant-Drug Relationships

Although Variant Annotation IDs do not overlap across the annotation datasets, the same biological relationship may be represented by different annotation records.

To investigate this possibility, records are compared using the combination of Gene, Variant/Haplotypes, and Drug(s).

This analysis tests whether the datasets contain overlapping biological relationships despite having distinct Variant Annotation IDs.

In [988]:
# Columns used to construct the biological relationship key.
relationship_columns = [
    "gene",
    "variant_haplotypes",
    "chemicals"
]
# Create standardized copies of these fields for comparison.
for df in [drug, phenotype, functional]:
    for column in relationship_columns:
        df[f"{column}_normalized"] = (
            df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower())

In [989]:
# Create a composite key representing a gene–variant–drug relationship.
for df in [drug, phenotype, functional]:
    df["gene_variant_drug_key"] = (
        df["gene_normalized"]
        + "|"
        + df["variant_haplotypes_normalized"]
        + "|"
        + df["chemicals_normalized"])

In [990]:
# Create sets of gene–variant–drug relationships.
drug_relationships = set(drug["gene_variant_drug_key"])
phenotype_relationships = set(phenotype["gene_variant_drug_key"])
functional_relationships = set(functional["gene_variant_drug_key"])

# Calculate pairwise overlap.
print(
    "Drug ↔ Phenotype:",
    len(drug_relationships & phenotype_relationships)
)
print(
    "Drug ↔ Functional:",
    len(drug_relationships & functional_relationships)
)
print(
    "Phenotype ↔ Functional:",
    len(phenotype_relationships & functional_relationships))

Drug ↔ Phenotype: 1438
Drug ↔ Functional: 130
Phenotype ↔ Functional: 141


In [991]:
# Find gene–variant–drug relationships shared by Drug and Phenotype datasets.
shared_drug_phenotype = (
    drug_relationships
    & phenotype_relationships
)
# Display several examples from each dataset.
drug_examples = (
    drug[drug["gene_variant_drug_key"].isin(shared_drug_phenotype)]
    [["variant_annotation_id", "gene", "variant_haplotypes", "chemicals"]]
    .head(10)
)
phenotype_examples = (
    phenotype[phenotype["gene_variant_drug_key"].isin(shared_drug_phenotype)]
    [["variant_annotation_id", "gene", "variant_haplotypes", "chemicals"]]
    .head(10)
)
display(drug_examples)
display(phenotype_examples)

,variant_annotation_id,gene,variant_haplotypes,chemicals
4,1448257202,CYP2C9,"CYP2C9*1, CYP2C9*3",warfarin
8,1448997750,CYP2B6,"CYP2B6*1, CYP2B6*18",efavirenz
13,982047744,CYP2D6,CYP2D6*1xN,codeine
16,1449191920,CFTR,rs113993960,ivacaftor / lumacaftor
22,1449192352,CFTR,rs113993960,ivacaftor / lumacaftor
26,1451308040,NAT2,NAT2 slow acetylator,isoniazid
30,1451308160,CYP2B6,CYP2B6 poor metabolizer,efavirenz
32,1453076880,CYP2C19,CYP2C19*1,lansoprazole
44,1453089269,CYP3A4,"CYP3A4*1, CYP3A4*22",lurbinectedin
45,1453085302,CYP3A5,"CYP3A5*1, CYP3A5*3",tacrolimus


,variant_annotation_id,gene,variant_haplotypes,chemicals
6,1453086760,CYP2B6,rs2279343,cyclophosphamide
8,827815626,CYP3A4,rs2740574,tacrolimus
20,1447990348,DPYD,DPYD deficiency,fluorouracil
23,1184515291,CYP3A5,"CYP3A5*1, CYP3A5*3",tacrolimus
24,1296599341,CYP2D6,"CYP2D6*5, CYP2D6*10",risperidone
25,1452485084,ABCG2,rs2231142,gefitinib
32,1449170079,HLA-B,HLA-B*58:01,allopurinol
33,1447519284,CYP2C9,"CYP2C9*1, CYP2C9*2, CYP2C9*3",warfarin
34,978616289,VKORC1,rs9923231,warfarin
41,1184515468,CYP3A5,"CYP3A5*1, CYP3A5*3",tacrolimus


*Interpretation*

Variant Annotation IDs are unique within each of the three annotation datasets, but no identifiers are shared between datasets. Therefore, Variant Annotation ID cannot be used as a direct cross-dataset merge key for these files.

However, substantial overlap exists when records are compared using Gene, Variant/Haplotypes, and Drug(s). This indicates that the datasets can contain different annotation records describing the same underlying gene–variant–drug relationship.

This distinction is important: the Variant Annotation ID appears to identify individual annotation records within the datasets, whereas the underlying biological relationship may be represented by multiple annotation records with different IDs.

*Conclusion*

Variant Annotation ID is not a suitable cross-dataset merge key for the three annotation datasets in this data release. Although it is complete and unique within each dataset, no IDs are shared across datasets. However, substantial overlap exists at the underlying gene–variant–drug level, suggesting that different annotation records can describe the same biological relationship. Further cardinality analysis is required before selecting a composite merge key.

### 7. Relationship Cardinality Analysis

The previous analysis showed that Variant Annotation ID is unique within the annotation datasets but does not overlap across them. However, the datasets contain substantial overlap at the gene–variant–drug level.

This section investigates the cardinality of those shared biological relationships. Specifically, it examines how many records correspond to the same gene–variant–drug combination within each dataset and whether additional fields are required to distinguish individual records.

Understanding these relationships is necessary before selecting a composite merge key because an inappropriate key can produce unintended many-to-many joins and inflate the number of observations in the integrated dataset.

#### 7.1 Candidate Biological Relationship

The initial candidate relationship is:

**Gene + Variant/Haplotypes + Drug(s)**

This combination is being investigated because it represents the biological relationship between a genetic factor and a medication that is central to the pharmacogenomic use case.

This is a candidate relationship rather than a confirmed merge key. Its uniqueness and consistency across datasets must be evaluated empirically.

#### 7.2 Verify Harmonized Relationship Fields

The datasets were previously standardized during Dataset Schema Harmonization so that equivalent biological concepts use consistent column names. Before constructing a candidate composite relationship, the harmonized column names are verified across all four datasets.

The candidate biological relationship consists of:
- `variant_haplotypes`
- `gene`
- `chemicals`

In [992]:
# Display the columns used to represent the candidate biological relationship in each harmonized dataset.

relationship_columns = [
    "variant_haplotypes",
    "gene",
    "chemicals"
]
datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(name)
    for column in relationship_columns:
        print(f"  {column}: {column in df.columns}")
    print()

Clinical Variants
  variant_haplotypes: True
  gene: True
  chemicals: True

Variant Drug Annotations
  variant_haplotypes: True
  gene: True
  chemicals: True

Variant Phenotype Annotations
  variant_haplotypes: True
  gene: True
  chemicals: True

Variant Functional Assay Annotations
  variant_haplotypes: True
  gene: True
  chemicals: True



#### 7.3 Build Four-Way Candidate Relationship

In [993]:
def create_relationship_key(df, columns):
    """
    Create a standardized composite key from the specified biological relationship columns.

    The original columns are not modified. Values are converted to
    strings, whitespace is removed, and text is converted to lowercase
    for comparison purposes.
    """
    standardized = (
        df[columns]
        .fillna("")
        .astype(str)
        .apply(lambda column: column.str.strip().str.lower())
    )
    return standardized.astype(str).agg("|".join, axis=1)

In [994]:
relationship_columns = [
    "variant_haplotypes",
    "gene",
    "chemicals"
]
for name, df in datasets.items():
    df["variant_gene_chemical_key"] = create_relationship_key(
        df,
        relationship_columns)

#### 7.4 Number of Unique Gene-Variant-Drug Relationships

The number of unique candidate relationships is compared with the number of records in each dataset. If the number of records exceeds the number of unique relationships, multiple records describe the same candidate relationship.

In [995]:
# Compare total records with unique gene–variant–drug relationships.
cardinality_summary = pd.DataFrame({
    "Dataset": [
        "Clinical Variants"
        "Variant Drug Annotations",
        "Variant Phenotype Annotations",
        "Variant Functional Assays"
    ],
    "Total Records": [
        len(clinical),
        len(drug),
        len(phenotype),
        len(functional)
    ],
    "Unique Gene-Variant-Drug Relationships": [
        clinical["gene_variant_chemical_key"].nunique(),
        drug["gene_variant_chemical_key"].nunique(),
        phenotype["gene_variant_chemical_key"].nunique(),
        functional["gene_variant_chemical_key"].nunique()
    ]})
cardinality_summary

KeyError: 'gene_variant_chemical_key'

In [ ]:
clinical_relationship_counts = (
    clinical["gene_variant_chemical_key"]
    .value_counts()
)
drug_relationship_counts = (
    drug["gene_variant_chemical_key"]
    .value_counts()
)
phenotype_relationship_counts = (
    phenotype["gene_variant_chemical_key"]
    .value_counts()
)
functional_relationship_counts = (
    functional["gene_variant_chemical_key"]
    .value_counts()
)

In [ ]:
print("Clinical Variants:")
print(clinical_relationship_counts.describe())

print("\nDrug Annotations:")
print(drug_relationship_counts.describe())

print("\nPhenotype Annotations:")
print(phenotype_relationship_counts.describe())

print("\nFunctional Assays:")
print(functional_relationship_counts.describe())

Drug Annotations:
count    8251.000000
mean        1.572537
std         4.022563
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max       197.000000
Name: count, dtype: float64

Phenotype Annotations:
count    8983.000000
mean        1.613047
std         3.049743
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        92.000000
Name: count, dtype: float64

Functional Assays:
count    1841.000000
mean        1.169473
std         0.713657
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        15.000000
Name: count, dtype: float64
